## 0. Imports

In [1]:
import json
import torch
import os
import cv2
import re

import pandas as pd
import numpy as np


from PIL import Image
from concurrent.futures import ProcessPoolExecutor
from table_metric import TEDS
from pathlib import Path
from tqdm import tqdm

import torchvision.models as models

from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
import torch.nn.functional as F
import torchvision.transforms.functional as TF




In [2]:
import torch
torch.cuda.empty_cache()

In [3]:
from paddleocr import PaddleOCRVL

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


## 1. Benchmark preparation

Для создания бенчмарка возьмем из каждой частей датасета (Fintabnet, Marketing, PubTabNet, Sparse) по 250 тестовых таблиц. Каждую из которых повернем на 90, 180 или 270 градусов, в зависимости от индекса. Также извлечём html код из аннотаций к датасету.

In [ ]:
def format_html(ann: dict) -> str:
    r"""
    Formats annotations in html code (from https://github.com/IBM/SynthTabNet/blob/main/synthtabnet_demo.ipynb)
    """
    cells = ann["html"]["cells"]
    tokens = ann["html"]["structure"]["tokens"]
    table = []
    cell_idx = 0
    for token in tokens:
        if token.startswith("<td"):
            cell = cells[cell_idx]
            content = "".join(cell["tokens"])
            cell_idx += 1
        if token == "</td>":
            table.append(content)
        table.append(token)
    table_str = "".join(table)

    html_code = """<html>
    <head>
    <meta charset="UTF-8">
    <style>
    table, th, td {
      border: 1px solid black;
      font-size: 10px;
    }
    </style>
    </head>
    <body>
    <table frame="hsides" rules="groups" width="100%%">
    %s
    </table>
    </body>
    </html>""" % "".join(table_str)
    return html_code

In [ ]:


def fix_and_load_json(line):
    line = re.sub(r',\s*([\]}])', r'\1', line)
    return json.loads(line)

def process_benchmark():
    base_dir = "data"
    output_dir = "benchmark_1000"
    subsets = ["fintabnet", "marketing", "pubtabnet", "sparse"]

    os.makedirs(f"{output_dir}/images", exist_ok=True)
    os.makedirs(f"{output_dir}/ground_truth", exist_ok=True)

    total_processed = 0
    gt_dict = {}

    for subset in subsets:
        subset_dir = os.path.join(base_dir, subset)

        jsonl_path = os.path.join(subset_dir, "synthetic_data.jsonl")
        print(jsonl_path)

        if not os.path.exists(jsonl_path):
            print(f"Warning: {jsonl_path} not found. Ensure paths are correct.")
            continue
        line_count = 0

        test_samples = []
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                line_count += 1
                try:
                    data = fix_and_load_json(line)
                    if data.get("split") == "test":
                        test_samples.append(data)
                except json.JSONDecodeError as e:
                    print(f"Ошибка в строке: {e}")
                    start = max(0, e.pos - 50)
                    end = min(len(line), e.pos + 50)
                    print("Контекст ошибки:")
                    print(line[start:end])
                    print(" " * (e.pos - start) + "^--- ОШИБКА ЗДЕСЬ")
                    continue

        test_samples.sort(key=lambda x: x["filename"])

        selected_samples = test_samples[:250]

        for index, sample in enumerate(selected_samples):
            filename = sample["filename"]
            img_path = os.path.join(subset_dir, "images", "test", filename)
            if not os.path.exists(img_path):
                print(1)
                continue

            img = cv2.imread(img_path)
            if img is None:
                print(2)
                continue

            rem = index % 4
            if rem == 0:
                rotated_img = img # No rotation
            elif rem == 1:
                rotated_img = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
            elif rem == 2:
                rotated_img = cv2.rotate(img, cv2.ROTATE_180)
            elif rem == 3:
                rotated_img = cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE) # 270 CW

            new_filename = f"{subset}_{filename}"
            cv2.imwrite(os.path.join(output_dir, "images", new_filename), rotated_img)

            # Task 1.4: Ground-Truth Table Representation
            gt_html = format_html(sample)
            gt_dict[new_filename] = {
                "subset": subset,
                "original_index": index,
                "rotation_applied": rem * 90,
                "html": gt_html
            }
            total_processed += 1

    with open(os.path.join(output_dir, "ground_truth", "gt_benchmark.json"), 'w', encoding='utf-8') as f:
        json.dump(gt_dict, f, indent=4)

    print(f"Benchmark generation complete. Total samples: {total_processed}/1000")

if __name__ == "__main__":
    process_benchmark()


Для оценки правильности парсинга используются TEDS и TEDS-sructs метрики. Я адаптировал предложенный скрипт, чтобы он считал средний TEDS и TEDS-struct по всем предсказаниям.

In [13]:


def evaluate_benchmark(predictions_path, gt_path, pred_file=True, gt_file=True):
    if gt_file:
        with open(gt_path, 'r', encoding='utf-8') as f:
            # Ожидаемый формат: {"filename.png": {"html": "<html>...</html>", ...}}
            gt_data = json.load(f)
    else:
        gt_data = gt_path

    
    if pred_file:
        with open(predictions_path, 'r', encoding='utf-8') as f:
            # Ожидаемый формат: {"filename.png": "<html>...</html>"}
            pred_data = json.load(f)
    else:
        pred_data = predictions_path
    

    print("Вычисление метрики TEDS (Full)...")
    teds_evaluator = TEDS(structure_only=False, n_jobs=1) 
    teds_scores_dict = teds_evaluator.batch_evaluate(pred_data, gt_data)

    print("Вычисление метрики TEDS-Struct (Structure Only)...")
    teds_struct_evaluator = TEDS(structure_only=True, n_jobs=1)
    teds_struct_scores_dict = teds_struct_evaluator.batch_evaluate(pred_data, gt_data)


    avg_teds = sum(teds_scores_dict.values()) / len(teds_scores_dict) if teds_scores_dict else 0.0
    avg_teds_struct = sum(teds_struct_scores_dict.values()) / len(teds_struct_scores_dict) if teds_struct_scores_dict else 0.0

    print("\n--- Финальные результаты ---")
    print(f"Средний TEDS:        {avg_teds:.4f}")
    print(f"Средний TEDS-Struct: {avg_teds_struct:.4f}")

## 2. Оценка модели на бенчмарке

- PaddleOCR-VL-1.5-0.9B

In [ ]:
import sys 
!{sys.executable} -m pip install paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu129/
!{sys.executable} -m pip install -U "paddleocr[doc-parser]"

In [15]:

# NVIDIA GPU
pipeline = PaddleOCRVL()


Creating model: ('PP-DocLayoutV3', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-DocLayoutV3`.
Creating model: ('PaddleOCR-VL-1.5-0.9B', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PaddleOCR-VL-1.5`.
Loading configuration file /root/.paddlex/official_models/PaddleOCR-VL-1.5/config.json
Loading weights file /root/.paddlex/official_models/PaddleOCR-VL-1.5/model.safetensors
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use G

**Сравним, как модель парсит повернутые и неповернутые таблицы**

In [20]:
test_output = pipeline.predict(["benchmark_1000/images/fintabnet_image_000029_1634629328.565314.png",
                              "benchmark_1000/images/fintabnet_image_000044_1634629328.589694.png"])
html_tables = [test_output[0].str['res']['parsing_res_list'][0]['block_content'],
               test_output[1].str['res']['parsing_res_list'][0]['block_content']]

In [21]:
html_tables[0]

'<table><tr><td>In thousands</td><td>Total deferred</td><td>8-K</td><td>Period</td><td>n/a</td><td>Business</td></tr><tr><td>Debt securities</td><td>$ \\dagger $</td><td>-</td><td></td><td></td><td></td></tr><tr><td>Foreign exchange</td><td>Other Benefits</td><td>North America</td><td>Minority interest</td><td>Hedge funds</td><td></td></tr><tr><td>Foreign exchange contracts</td><td>78.6%</td><td>46.86%</td><td>10.38%</td><td>90.64%</td><td>80.74%</td></tr><tr><td>In Thousands</td><td></td><td>80.75%</td><td>24.70%</td><td></td><td>43.22%</td></tr><tr><td>Dollars</td><td>79.22%</td><td>82.99%</td><td>9.43%</td><td>71.20%</td><td>96.23%</td></tr><tr><td>Lease</td><td>83.56%</td><td>42.31%</td><td>37.57%</td><td>55.72%</td><td>16.76%</td></tr><tr><td>China</td><td>42.1%</td><td>88.26%</td><td>57.41%</td><td>54.19%</td><td></td></tr><tr><td>Entergy Mississippi</td><td>44.76%</td><td>52.5%</td><td>73.52%</td><td></td><td>71.14%</td></tr><tr><td>Federal</td><td>99.70%</td><td>94.51%</td><td>

![таблица из бенчмарка](data/fintabnet_image_000029_1634629328.565314.png)

'<table><tr><td>In thousands</td><td>Total deferred</td><td>8-K</td><td>Period</td><td>n/a</td><td>Business</td></tr><tr><td>Debt securities</td><td>$ \\dagger $</td><td>-</td><td></td><td></td><td></td></tr><tr><td>Foreign exchange</td><td>Other Benefits</td><td>North America</td><td>Minority interest</td><td>Hedge funds</td><td></td></tr><tr><td>Foreign exchange contracts</td><td>78.6%</td><td>46.86%</td><td>10.38%</td><td>90.64%</td><td>80.74%</td></tr><tr><td>In Thousands</td><td></td><td>80.75%</td><td>24.70%</td><td></td><td>43.22%</td></tr><tr><td>Dollars</td><td>79.22%</td><td>82.99%</td><td>9.43%</td><td>71.20%</td><td>96.23%</td></tr><tr><td>Lease</td><td>83.56%</td><td>42.31%</td><td>37.57%</td><td>55.72%</td><td>16.76%</td></tr><tr><td>China</td><td>42.1%</td><td>88.26%</td><td>57.41%</td><td>54.19%</td><td></td></tr><tr><td>Entergy Mississippi</td><td>44.76%</td><td>52.5%</td><td>73.52%</td><td></td><td>71.14%</td></tr><tr><td>Federal</td><td>99.70%</td><td>94.51%</td><td>49.17%</td><td>19.8%</td><td>19.35%</td></tr><tr><td>Outstanding at beginning of year</td><td>87.86%</td><td>37.75%</td><td>24.93%</td><td>60.74%</td><td></td></tr><tr><td>Fiscal Years Ended</td><td>10.68%</td><td></td><td>78.25%</td><td>32.96%</td><td>0.59%</td></tr><tr><td>U.S.</td><td>3.90%</td><td>70.25%</td><td>31.94%</td><td>78.76%</td><td>4.98%</td></tr><tr><td>Goodwill</td><td>98.53%</td><td>51.66%</td><td>57.89%</td><td></td><td>0.1%</td></tr><tr><td>Retained earnings</td><td>86.16%</td><td>80.70%</td><td>31.32%</td><td></td><td>45.50%</td></tr><tr><td>Current</td><td>37.66%</td><td>41.21%</td><td>88.93%</td><td>76.56%</td><td></td></tr></table>'

С неповернутой таблицей модель отлично справилась.

---

In [22]:
html_tables[1]

'Operating Profit'

![таблица из бенчмарка 2](data/fintabnet_image_000044_1634629328.589694.png)

С повернутой же - все ломается

---

- Cохраним предсказания модели в json файл

In [16]:
otn_path = 'benchmark_1000/images/'
bench_images = os.listdir(otn_path)
bench_images_crop = bench_images[0:5] + bench_images[250: 255] + bench_images[500:505] + bench_images[750 : 755]
bench_images_crop = [otn_path + path for path in bench_images_crop]

In [17]:

output = pipeline.predict(bench_images_crop)

res_to_json = {}
for res in tqdm(output):
    res_dict = res.str # xd 
    image_path = res_dict['res']['input_path']
    filename = image_path.split('/')[-1]
    html_table = res_dict['res']['parsing_res_list'][0]['block_content']
    
    res_to_json[filename] = html_table

with open('results.json', 'w', encoding='utf-8') as f:
    json.dump(res_to_json, f, ensure_ascii=False, indent=4)

    

100%|██████████| 20/20 [00:00<00:00, 4357.04it/s]


- И посчитаем TEDS и TEDS-Struct метрики

In [18]:
evaluate_benchmark('results.json', 'benchmark_1000/ground_truth/gt_benchmark.json')

Вычисление метрики TEDS (Full)...


100%|██████████| 20/20 [00:03<00:00,  5.39it/s]


Вычисление метрики TEDS-Struct (Structure Only)...


100%|██████████| 20/20 [00:02<00:00,  8.69it/s]


--- Финальные результаты ---
Средний TEDS:        0.6446
Средний TEDS-Struct: 0.7097


## 3. Auxiliary Module: Rotation-Aware Preprocessing

Как было показано выше, модели плохо справляются с таблицами, которые повернуты на неправильный угол. Поэтому было бы хорошо понимать на какой угол повернута картинка, чтобы можно было повернуть ее в обратном направлении. Можно решить задачу классификации изображений, где классом будет один из четырех углов (0, 90, 180, 270), а затем подать на вход модели-парсеру исправленное изображение.

- Сделаю датасет поменьше (50к изображений), файл с аннотациями и применю повороты к таблицам.

In [ ]:

NEW_TRAIN_DIR = Path("/root/project/data_train")
NEW_TRAIN_DIR.mkdir(parents=True, exist_ok=True)

def rotate_and_copy_single(item_tuple):
    index, item = item_tuple
    img_path = item['full_path']
    label = index % 4
    

    save_path = NEW_TRAIN_DIR / f"{item['subset']}_{item['filename']}"
    
    try:
        with Image.open(img_path) as img:
            img = img.convert('RGB')
            
            if label == 1:
                img = img.transpose(Image.Transpose.ROTATE_270)
            elif label == 2:
                img = img.transpose(Image.Transpose.ROTATE_180)
            elif label == 3:
                img = img.transpose(Image.Transpose.ROTATE_90)
            
            img.save(save_path)
            
        return {
            "filename": save_path.name, 
            "subset": item['subset'],
            "label": label
        }
    except Exception as e:
        return None

def process_synthtabnet_subset(base_dir, split_name, limit=50000):
    base_path = Path(base_dir)
    subsets = ['fintabnet', 'marketing', 'pubtabnet', 'sparse']
    all_images = []

    print(f"Сбор путей для {split_name}...")
    for subset in subsets:
        split_dir = base_path / subset / "images" / split_name
        if split_dir.exists():
            files = list(split_dir.glob("*.png")) + list(split_dir.glob("*.jpg"))
            for f in files:
                all_images.append({"full_path": f, "filename": f.name, "subset": subset})
    
    all_images.sort(key=lambda x: x['filename'])
    selected_images = all_images[-limit:]
    
    print(f"Запуск копирования и поворота {len(selected_images)} изображений в {NEW_TRAIN_DIR}...")
    
    with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
        results = list(tqdm(executor.map(rotate_and_copy_single, enumerate(selected_images)), total=len(selected_images)))
    

    final_data = [r for r in results if r is not None]
    df = pd.DataFrame(final_data)
    csv_name = "dataset.csv"
    df.to_csv(csv_name, index=False)
    print(f"Готово! Данные в {NEW_TRAIN_DIR}, аннотации в {csv_name}")

 
process_synthtabnet_subset('/root/project/data', 'train', limit=50000)

In [ ]:
from sklearn.model_selection import train_test_split

df = pd.read_csv('dataset.csv')

train_df, test_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42, 
    stratify=df['label']
)

train_df.to_csv('train.csv', index=False)
test_df.to_csv('test.csv', index=False)

- Создадим класс Dataset 

In [4]:


class SquarePad:
    def __call__(self, image):
        w, h = image.size
        max_wh = max(w, h)
        hp = int((max_wh - w) / 2)
        vp = int((max_wh - h) / 2)
        padding = (hp, vp, max_wh - w - hp, max_wh - h - vp)
        return TF.pad(image, padding, fill=255, padding_mode='constant')

class DocDataset(Dataset):
    def __init__(self, csv_file, root_dir, split, transform=None):
        self.df = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.split = split
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        img_path = os.path.join(self.root_dir, row['filename'])
        
        image = Image.open(img_path).convert('RGB')
        label = int(row['label'])

        if self.transform:
            image = self.transform(image)

        return image, label





In [5]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    SquarePad(),                 
    transforms.Resize((512, 512)), 
    transforms.ToTensor(),        
    transforms.Normalize(mean=mean, std=std) 
])


In [6]:
train_dataset = DocDataset('train.csv', 'data_train', 'train', transform=transform)
test_dataset = DocDataset('test.csv', 'data_train', 'test', transform=transform)

trainloader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4)
testloader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=4)

In [7]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
device

device(type='cuda')

In [8]:
def train_one_epoch(model, loss_func, optimizer, train_dl):
    model.train() 
    total_loss = 0.0

    for xb, yb in tqdm(train_dl, desc="Training"):
        xb, yb = xb.to(device), yb.to(device)

        preds = model(xb)
        loss = loss_func(preds, yb)

        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    return total_loss / len(train_dl)


def validate(model, loss_func, valid_dl):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for xb, yb in tqdm(valid_dl, desc="Validation"):
            xb, yb = xb.to(device), yb.to(device)

            preds = model(xb)
            loss = loss_func(preds, yb)
            total_loss += loss.item()

            _, predicted_labels = torch.max(preds, dim=-1)
            total_correct += (predicted_labels == yb).sum().item()
            total_samples += xb.size(0)

    avg_loss = total_loss / len(valid_dl)
    accuracy = total_correct / total_samples
    return avg_loss, accuracy


def fit(epochs, model, loss_func, optimizer, train_dl, valid_dl):
    train_losses = []
    val_losses = []
    valid_accuracies = []

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")

        train_loss = train_one_epoch(model, loss_func, optimizer, train_dl)

        val_loss, val_acc = validate(model, loss_func, valid_dl)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        valid_accuracies.append(val_acc)

        print(f"Epoch {epoch+1} -> "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_acc:.4f}")

    return train_losses, val_losses, valid_accuracies

In [9]:
resnet18 = models.resnet18()

In [10]:
resnet18

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [11]:
num_features = 512
resnet18.fc = nn.Linear(num_features, 4)
resnet18 = resnet18.to(device)

In [12]:
resnet18

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet18.parameters(), lr=0.001)


In [14]:
train_losses, val_losses, valid_accuracies = fit(5, resnet18, criterion, optimizer, trainloader, testloader)


Epoch 1/5


Validation: 100%|██████████| 79/79 [00:39<00:00,  1.98it/s]


Epoch 1 -> Train Loss: 0.1616 | Val Loss: 0.0080 | Val Acc: 0.9977

Epoch 2/5


Validation: 100%|██████████| 79/79 [00:39<00:00,  1.98it/s]


Epoch 2 -> Train Loss: 0.0187 | Val Loss: 0.0067 | Val Acc: 0.9989

Epoch 3/5


...

Epoch 3 -> Train Loss: 0.0139 | Val Loss: 0.0170 | Val Acc: 0.9949

Epoch 4/5


Validation: 100%|██████████| 79/79 [00:39<00:00,  1.99it/s]


Epoch 4 -> Train Loss: 0.0060 | Val Loss: 0.0025 | Val Acc: 0.9996

Epoch 5/5


Validation: 100%|██████████| 79/79 [00:39<00:00,  1.99it/s]

Epoch 5 -> Train Loss: 0.0173 | Val Loss: 0.0008 | Val Acc: 0.9998


In [ ]:
torch.save(resnet18, 'doc_parser_model.pth')

---

## 4. Final Pipeline and Evaluation

**Теперь, после обучения классификатора, объеденим обе модели и посмотрим, улучшится ли качество.**


In [7]:

class SuperTableParser:
    def __init__(self, objects, transform):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model_classificator = torch.load('doc_parser_model.pth', weights_only=False).to(self.device)
        self.model_classificator.eval()
        
        self.model_parser = PaddleOCRVL() 
        self.objects = objects
        self.transform = transform
        self.rotation_map = {
            0: lambda img: img,                                   
            1: lambda img: img.rotate(270, expand=True),         
            2: lambda img: img.rotate(180, expand=True), 
            3: lambda img: img.rotate(90, expand=True)
        }
        
    def predict(self) -> dict:
        processed_images = []
        
        for path in tqdm(self.objects, desc="Classifying"):
            img = Image.open(path).convert('RGB')
            
            input_tensor = self.transform(img).unsqueeze(0).to(self.device)

            with torch.no_grad(): 
                output = self.model_classificator(input_tensor)
                predicted_class = torch.argmax(output, dim=1).item()

            action = self.rotation_map.get(predicted_class, lambda x: x)
            img_rotated = action(img)
            
            img_array = np.array(img_rotated)
            processed_images.append(img_array)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        output_data = self.model_parser.predict(processed_images)

        res_to_json = {}
        for i, res in enumerate(output_data):
            res_dict = res.str 
            html_table = res_dict['res']['parsing_res_list'][0]['block_content']
            image_path = self.objects[i]
            filename = image_path.split('/')[-1]
            res_to_json[filename] = html_table 
            
        return res_to_json
        

In [9]:
stp = SuperTableParser(bench_images_crop, transform)

/opt/conda/lib/python3.11/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-DocLayoutV3', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-DocLayoutV3`.
Creating model: ('PaddleOCR-VL-1.5-0.9B', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PaddleOCR-VL-1.5`.
Loading configuration file /root/.paddlex/official_models/PaddleOCR-VL-1.5/config.json
Loading weights file /root/.paddlex/official_models/PaddleOCR-VL-1.5/model.safetensors
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_head

In [10]:
super_preds = stp.predict()

Classifying: 100%|██████████| 20/20 [00:00<00:00, 36.65it/s]
/opt/conda/lib/python3.11/site-packages/paddle/tensor/creation.py:1152: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach(), rather than paddle.to_tensor(sourceTensor).
  return tensor(


In [14]:
evaluate_benchmark(super_preds, 'benchmark_1000/ground_truth/gt_benchmark.json', pred_file=False, gt_file=True)

Вычисление метрики TEDS (Full)...


100%|██████████| 20/20 [00:04<00:00,  4.99it/s]


Вычисление метрики TEDS-Struct (Structure Only)...


100%|██████████| 20/20 [00:02<00:00,  8.01it/s]


--- Финальные результаты ---
Средний TEDS:        0.7689
Средний TEDS-Struct: 0.8245


**Вывод: Строить структуру таблицы с таким пайплайном модели стало легче.**